# Trailing plot (logL time series)

Shows the logL as a function of orbital phase and radial velocity at a fixed Kp.
A detection appears as a tilted streak in the trailing map (the planet's RV changes
with orbital phase) and a clear peak in the integrated 1D logL profile.

**Workflow:**
1. Load logl grid results (pre-computed NPZ files)
2. Inspect alpha_frac (which exposures see the planet signal)
3. Generate the trailing map for each visit
4. Overlay the combined 1D profile across visits

All the heavy computation is done by `run_starships_logl_grid` on the cluster.
This notebook is for analysis only.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import starships.logl_grid as lg
from starships.plotting_fcts import plot_trailing_map

## 1. Paths and parameters

Set `path_results` to the directory where the NPZ files are stored,
and `file_stem` to the model filename stem used when running the grid.

In [ ]:
path_results = Path.home() / 'scratch/DataAnalysis/SPIRou/logl_grids/'

# Glob pattern for the visit NPZ files.
# run_starships_logl_grid encodes grid parameters in the filename:
#   {model_stem}_kp{min}_{max}_{step}_rv{min}_{max}_{step}[_noalpha]_visit*.npz
# Edit the pattern below to match your run.
file_glob = 'take3_HRR_modif_disso_31254924_all_kp*_visit*.npz'

# Reference Kp and expected vsys for this planet
kp_ref = 227.15      # km/s
rv_expected = 0.0    # km/s — adjust to known Vsys

# Half-widths of the noise and peak windows around rv_expected
noise_rv_width = 15.   # km/s  — points outside ±noise_rv_width define the baseline
peak_rv_width  =  2.   # km/s  — points inside ±peak_rv_width are the peak

## 2. Load the logl grid results

In [ ]:
files = sorted(path_results.glob(file_glob))
print(f'Found {len(files)} visit file(s):')
for f in files:
    print(' ', f.name)

# Load all visits into module globals
lg.load_logl_results(files)

## 3. Inspect alpha_frac

`alpha_frac` is the fraction of the total planet signal received during each
exposure. It equals 1 during full eclipse/transit and 0 when the planet is
hidden. Exposures with `alpha_frac > 0.5` are 'out of eclipse' — these are
the only ones that contribute a meaningful planet signal.

In [ ]:
%matplotlib inline

alpha_frac = lg._loaded_extra['alpha_frac']
phase      = lg._loaded_extra.get('phase', np.arange(len(alpha_frac)))

# kind_trans is saved as a 1-element string array
kind_trans = str(lg._loaded_extra.get('kind_trans', np.array(['transmission']))[0])

# For emission the signal is outside the eclipse (alpha_frac peaks near phase 0.5).
# For transmission the signal is during the transit (alpha_frac peaks near phase 0).
signal_label = 'in-eclipse threshold' if kind_trans == 'emission' else 'in-transit threshold'

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(phase, alpha_frac, 'o-', markersize=4)
ax.axhline(0.5, linestyle='--', color='gray', label=signal_label)
ax.set_xlabel('Orbital Phase')
ax.set_ylabel(r'$\alpha_\mathrm{frac}$')
ax.set_title(f'kind_trans = {kind_trans!r}')
ax.legend()
plt.tight_layout()

In [ ]:
(idx_signal,) = np.nonzero(alpha_frac > 0.5)
print(f'Exposures with planet signal: {len(idx_signal)} / {len(alpha_frac)}')

## 4. Trailing map

The trailing map shows the normalised logL at a fixed Kp as a function of
orbital phase and vsys.  A planet signal appears as a tilted streak because
its RV changes with orbital phase at a rate set by Kp.

Normalisation: per-exposure baseline (median over the `noise_rv_limits` window)
is subtracted and divided by the per-exposure standard deviation over the same
window, putting each exposure on a common S/N scale.

In [ ]:
# Locate the nearest Kp in the grid (argmin is more robust than searchsorted,
# which returns the insertion point and can land on the wrong neighbour).
i_kp = int(np.argmin(np.abs(lg.kp_axis - kp_ref)))
print(f'Using Kp = {lg.kp_axis[i_kp]:.2f} km/s (requested {kp_ref:.2f})')

# Noise window centred on rv_expected
noise_rv_limits = (rv_expected - noise_rv_width, rv_expected + noise_rv_width)
peak_rv_limits  = (rv_expected - peak_rv_width,  rv_expected + peak_rv_width)
is_out_rv = (lg.vsys_axis < noise_rv_limits[0]) | (noise_rv_limits[1] < lg.vsys_axis)

In [ ]:
# Combined 1D profile (all visits, signal exposures only)
logl_1d_all = lg.get_logl(idx_exposure=idx_signal, alpha=1., sum_axis=(-2, -1))[:, i_kp]
logl_1d_all_norm = logl_1d_all - np.ma.median(logl_1d_all[is_out_rv])
logl_1d_all_norm /= np.ma.std(logl_1d_all_norm[is_out_rv])

In [ ]:
# Plot one trailing map per visit
for i_file, filename in enumerate(files, start=1):
    lg.load_logl_results([filename])

    visit_alpha = lg._loaded_extra['alpha_frac']
    phase_v = lg._loaded_extra.get('phase', None)
    if phase_v is None or len(phase_v) == 0:
        phase_v = np.arange(lg.N.shape[-2])

    (visit_signal,) = np.nonzero(visit_alpha > 0.5)

    contact_phases = lg._loaded_extra.get('contact_phases', None)
    if contact_phases is not None and len(contact_phases) == 4:
        phase_contacts = {
            '1_4': [float(contact_phases[0]), float(contact_phases[3])],
            '2_3': [float(contact_phases[1]), float(contact_phases[2])],
        }
    else:
        phase_contacts = None

    logl_map_ts = lg.get_logl(alpha=1., sum_axis=-1)[:, i_kp, :]

    logl_map_norm = (logl_map_ts
                     - np.ma.median(logl_map_ts[is_out_rv, :], axis=0)[None, :])
    logl_map_norm /= np.ma.std(logl_map_ts[is_out_rv, :], axis=0)[None, :]

    logl_1d = lg.get_logl(idx_exposure=visit_signal, alpha=1., sum_axis=(-2, -1))[:, i_kp]
    logl_1d_norm = logl_1d - np.ma.median(logl_1d[is_out_rv])
    logl_1d_norm /= np.ma.std(logl_1d_norm[is_out_rv])

    # Per-exposure fractional contribution to the combined logL at rv_expected:
    #   delta_i = ΔlogL_i(vsys_peak) = baseline-subtracted logL for exposure i
    #   contrib_i = delta_i / sum_j(delta_j over signal exposures)
    # This sums to 1 over idx_signal and shows which exposures drive the peak.
    # Set vsys_peak to None to skip this overlay.
    vsys_peak = rv_expected  # or: vsys_peak = None
    if vsys_peak is not None:
        i_vsys_peak = int(np.argmin(np.abs(lg.vsys_axis - vsys_peak)))
        delta = logl_map_ts[i_vsys_peak, :] - np.ma.median(logl_map_ts[is_out_rv, :], axis=0)
        logl_comb = float(np.sum(delta[visit_signal]))
        contrib = delta / logl_comb if logl_comb != 0 else None
    else:
        contrib = None

    fig, axes = plot_trailing_map(
        logl_map_norm, lg.vsys_axis, phase_v, logl_1d_norm,
        noise_rv_limits=noise_rv_limits,
        peak_rv_limits=peak_rv_limits,
        logl_1d_norm_all=logl_1d_all_norm,
        rv_expected=rv_expected,
        phase_contacts=phase_contacts,
        contrib=contrib,
    )
    fig.suptitle(f'Visit {i_file}: {filename.name}', y=1.02)
    plt.show()

# Reload all visits for subsequent analysis
lg.load_logl_results(files)

## 5. Options and customisation

You can restrict the analysis to specific orders (`idx_orders`) or explore
different logL prescriptions (`kind='G'` for Gibson, `kind='BL'` for Brogi & Line).

In [ ]:
# Example: subset of orders suspected of carrying the signal
# idx_orders_subset = [14, 15, 30, 31, 46, 47]

# Example: Gibson logL instead of Brogi & Line
# logl_1d_G = lg.get_logl(idx_exposure=out_of_eclipse, alpha=1., kind='G', sum_axis=(-2, -1))[:, i_kp]

# Example: CCF instead of logL
# ccf_1d = lg.get_ccf(idx_exposure=out_of_eclipse, kind='BL', sum_axis=(-2, -1))[:, i_kp]
print('See commented examples above.')